# 🏷️ Notebook 2: CRF Baseline

**Traditional NER approach using Conditional Random Fields**

This notebook:
1. Loads and preprocesses the Hindi NER dataset
2. Extracts handcrafted features (affixes, script type, context, Hindi-specific)
3. Trains a CRF model
4. Evaluates with entity-level P/R/F1
5. Inspects top features learned by CRF
6. Saves predictions for later comparison

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import warnings
warnings.filterwarnings('ignore')

from src.data_utils import load_hiner_dataset, format_for_crf, ID_TO_LABEL
from src.crf_model import HindiCRFModel
from src.evaluation import save_results

## 1. Load and Prepare Data

In [ ]:
# Load dataset
dataset = load_hiner_dataset()

# Convert to CRF format: lists of (tokens, labels)
train_sents, train_labels = format_for_crf(dataset['train'])

# Use 'validation' if available, else 'dev'
val_key = 'validation' if 'validation' in dataset else 'dev'
val_sents, val_labels = format_for_crf(dataset[val_key])

test_sents, test_labels = format_for_crf(dataset['test'])

print(f"Train: {len(train_sents)} sentences")
print(f"Val:   {len(val_sents)} sentences")
print(f"Test:  {len(test_sents)} sentences")

## 2. Inspect Features

Let's look at the features extracted for a sample sentence.

In [ ]:
from src.feature_extraction import sentence_to_features

# Pick a sentence with entities
sample_idx = 0
for i, labels in enumerate(train_labels):
    if any(l != 'O' for l in labels):
        sample_idx = i
        break

sample_sent = train_sents[sample_idx]
sample_labs = train_labels[sample_idx]
sample_features = sentence_to_features(sample_sent)

print(f"Sentence: {' '.join(sample_sent)}")
print(f"\nFeatures for first token '{sample_sent[0]}':")
for key, val in sorted(sample_features[0].items()):
    print(f"  {key:30s}: {val}")

## 3. Train CRF

In [ ]:
# Initialize CRF model
crf_model = HindiCRFModel(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    context_window=2,
)

# Train
train_results = crf_model.train(
    train_sents, train_labels,
    val_sents, val_labels,
)

## 4. Evaluate on Test Set

In [ ]:
# Evaluate on test set
test_metrics = crf_model.evaluate(test_sents, test_labels)

print("\n" + "=" * 60)
print("CRF — TEST SET RESULTS")
print("=" * 60)
print(f"\nOverall Precision: {test_metrics['overall_precision']:.4f}")
print(f"Overall Recall:    {test_metrics['overall_recall']:.4f}")
print(f"Overall F1:        {test_metrics['overall_f1']:.4f}")
print(f"\nDetailed Report:\n")
print(test_metrics['report_str'])

## 5. Top Features Learned by CRF

This is useful for your report — shows what the CRF considers important.

In [ ]:
top_features = crf_model.get_top_features(n=15)

for label, features in top_features.items():
    print(f"\n--- Top features for {label} ---")
    for feat_name, weight in features[:10]:
        print(f"  {weight:+8.4f}  {feat_name}")

## 6. Save Model and Predictions

In [ ]:
# Save model
crf_model.save('../models/crf_model.pkl')

# Save predictions for error analysis
crf_predictions = crf_model.get_predictions_with_tokens(test_sents, test_labels)

# Save metrics
save_results(test_metrics, '../results/crf_test_metrics.json')

# Save predictions
import json
with open('../results/predictions/crf_predictions.json', 'w', encoding='utf-8') as f:
    json.dump(crf_predictions, f, ensure_ascii=False, indent=2)
print("✓ CRF predictions saved")

## 7. Quick Sample Predictions

In [ ]:
# Show some predictions
print("\nSample CRF Predictions:")
print("=" * 60)

shown = 0
for pred in crf_predictions:
    # Only show sentences with entities
    if any(l != 'O' for l in pred['true_labels']):
        print(f"\nSentence: {' '.join(pred['tokens'])}")
        print(f"{'Token':20s} {'True':10s} {'Predicted':10s} {'':5s}")
        print("-" * 50)
        for t, true_l, pred_l in zip(pred['tokens'], pred['true_labels'], pred['pred_labels']):
            match = '✓' if true_l == pred_l else '✗'
            print(f"{t:20s} {true_l:10s} {pred_l:10s} {match}")
        shown += 1
        if shown >= 5:
            break